# Churn Predictor

In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle

In [2]:
data = pd.read_csv("../datasets/Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Preprocess Data
data.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# One-Hot Encode Geography column
geography_encoder = OneHotEncoder(sparse_output=False)
geo_encoder = geography_encoder.fit_transform(data[["Geography"]])
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [6]:
geography_encoder.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
geo_encoded_df = pd.DataFrame(geo_encoder, columns=geography_encoder.get_feature_names_out(["Geography"]))
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [8]:
# Combine columns
data.drop(columns=["Geography"], inplace=True)
data = pd.concat([data, geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
# Save the encoders and scalers
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("one_hot_encoder_geography.pkl", "wb") as file:
    pickle.dump(geography_encoder, file)

In [10]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
cat_cols = []
passthrough_cols = ['Gender', 'HasCrCard', 'IsActiveMember', 'Geography_France', 'Geography_Germany', 'Geography_Spain']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols), # drop='first' avoids dummy trap
        ('pass', 'passthrough', passthrough_cols)
    ]
)


In [11]:
# Divide datasets into dependant / independant
X = data.drop('Exited', axis=1)
y = data['Exited']

# perform a test-train split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Scale the features
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [12]:
# Save scaler

with open("sclaer.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

## ANN Implementation

We need to define the following things to create an ANN


1. we first need to initialise a Sequential Network (which will have layers, and we will perform forward/backward propagations)
2. For creating a hidden neuron, we create a Dense object
3. Each node will have an activation function associated to it (sigmoid, softmax, relu, tan h, leaky relu). It is generally a practice to use ReLU in hidden nodes and sigmoid/softmax in outputs. 
4. Optimiser - these are used in back propagation, as optimisers are used for updating the weights
5. Loss Function
6. Metrics
7. Training Information -> Logs (in a folder) -> Used by Tensorboard -> Visualisation

In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from datetime import datetime
import keras_tuner as kt

In [14]:
X_train.shape

(8000, 12)

In [15]:
# Build ANN Model
# X_train.shape will decide the number of input nodes
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(40, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 40)             │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            41 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 561 (2.19 KB)

 Trainable params: 561 (2.19 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [17]:
# Compile the model
# model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])
model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])

In [18]:
## Setup TensorBoard
log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tf_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [19]:
# Setup Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=50,restore_best_weights=True)

In [20]:
# Training the model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=500, callbacks=[tf_callback, early_stopping_callback])

Epoch 1/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 888us/step - accuracy: 0.8279 - loss: 0.3994 - val_accuracy: 0.8485 - val_loss: 0.3624
Epoch 2/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step - accuracy: 0.8504 - loss: 0.3557 - val_accuracy: 0.8550 - val_loss: 0.3528
Epoch 3/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 588us/step - accuracy: 0.8560 - loss: 0.3480 - val_accuracy: 0.8550 - val_loss: 0.3426
Epoch 4/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 592us/step - accuracy: 0.8604 - loss: 0.3456 - val_accuracy: 0.8595 - val_loss: 0.3435
Epoch 5/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step - accuracy: 0.8624 - loss: 0.3428 - val_accuracy: 0.8645 - val_loss: 0.3360
Epoch 6/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 587us/step - accuracy: 0.8604 - loss: 0.3379 - val_accuracy: 0.8685 - val_loss: 0.3293
Epoch 7/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 583us/step - accuracy: 0.8635 - loss: 0.3377 - val_accuracy: 0.8580 - val_loss: 0.3322
Epoch 8/500
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - accuracy: 0.8587 - loss: 0

In [21]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))

    # Allow tuner to decide how many hidden layers (e.g., between 1 and 3)
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)

    for i in range(num_layers):
        # Allow tuner to pick the number of neurons for each layer
        units = hp.Int(f'units_{i}', min_value=16, max_value=128, step=16)
        model.add(Dense(units=units, activation='relu'))

    # Output layer remains fixed
    model.add(Dense(1, activation='sigmoid'))

    # You can also tune the learning rate, or fix it to 0.001
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy']
    )
    return model

In [22]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective='val_loss',          # Try to minimize validation loss
    max_trials=10,                 # Test 10 different architecture combinations
    executions_per_trial=1,
    directory='tuner_dir',
    project_name='churn_tuning',
    overwrite=True                 # Overwrites previous runs if you re-execute
)

In [23]:
tuner.search(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,                   # Set a reasonable epoch cap; early stopping will handle the rest
    callbacks=[early_stopping_callback],
    batch_size=32
)

Trial 10 Complete [00h 00m 08s]
val_loss: 0.33109360933303833

Best val_loss So Far: 0.32873988151550293
Total elapsed time: 00h 01m 53s


In [24]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Optimal Number of Layers: {best_hp.get('num_layers')}")
for i in range(best_hp.get('num_layers')):
    print(f"Optimal Units in Layer {i}: {best_hp.get(f'units_{i}')}")
print(f"Optimal Learning Rate: {best_hp.get('learning_rate')}")

# 2. Get the best trained model directly
best_model = tuner.get_best_models(num_models=1)[0]

# 3. Evaluate the best model on test data
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

Optimal Number of Layers: 3
Optimal Units in Layer 0: 48
Optimal Units in Layer 1: 32
Optimal Units in Layer 2: 80
Optimal Learning Rate: 0.01
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.8690 - loss: 0.3287
Test Accuracy: 0.8690, Test Loss: 0.3287


/Users/vedanshdwivedi/ai_course/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


In [25]:
best_model.save("best_model.keras")